# Phase 2: Exploratory Data Analysis (German)
Datasets: RP-Mod & RP-Crowd (Rheinische Post), GermEval 2021 (Facebook)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
df_rp = pd.read_csv("../data/raw/rp_mod_crowd/RP-Mod-Crowd.csv")
df_ge_train = pd.read_csv("../data/raw/germeval2021/GermEval21_TrainData.csv")
df_ge_test = pd.read_csv("../data/raw/germeval2021/GermEval21_TestData.csv")

for name, df in [("RP-Mod-Crowd", df_rp), ("GermEval train", df_ge_train), ("GermEval test", df_ge_test)]:
    print(name, df.shape)
    print(list(df.columns), "\n")

## Columns and value ranges

In [ ]:
# Remove leftover index columns (in memory only, raw file stays unchanged)
df_rp = df_rp.drop(columns=["Unnamed: 0.1", "Unnamed: 0"])

count_cols = [col for col in df_rp.columns if "Count Crowd" in col]
df_rp[["Reject Newspaper", "Reject Crowd"] + count_cols].describe().T[["mean", "min", "max"]]

In [ ]:
# How often do moderators and crowd agree?
pd.crosstab(df_rp["Reject Newspaper"], df_rp["Reject Crowd"])

In [ ]:
df_ge_train["Sub1_Toxic"].value_counts()

## reasons for crowd rejection

In [ ]:
# Is "Reject Crowd" simply a majority vote (at least 3 of 5)?
majority = df_rp["Rejection Count Crowd"] >= 3
print(f"Reject Crowd equals majority vote: {(df_rp['Reject Crowd'] == majority).mean():.1%}")

toxic_cols = ["Sexism Count Crowd", "Racism Count Crowd", "Threat Count Crowd",
              "Insult Count Crowd", "Profanity Count Crowd"]
other_cols = ["Meta Count Crowd", "Advertisement Count Crowd"]

rejected = df_rp[df_rp["Reject Crowd"] == 1]
has_toxic = (rejected[toxic_cols] >= 3).any(axis=1)
has_other = (rejected[other_cols] >= 3).any(axis=1)

print(f"\nRejected by crowd: {len(rejected)}")
print(f"  with a toxic category (majority):       {has_toxic.sum()}")
print(f"  only meta/advertisement (majority):     {(~has_toxic & has_other).sum()}")
print(f"  no category reached a majority:         {(~has_toxic & ~has_other).sum()}")

In [ ]:
unclear = rejected[~has_toxic & ~has_other]

toxic_votes = unclear[toxic_cols].sum(axis=1)
other_votes = unclear[other_cols].sum(axis=1)

print(f"More toxic than meta/ad votes: {(toxic_votes > other_votes).mean():.1%}")
unclear[toxic_cols + other_cols].mean().round(2)

## Mapping RP-Crowd to the unified format
Labels are crowd vote shares (votes / 5). Rejected comments where meta/advertisement
votes are at least as frequent as toxic votes are excluded (not toxicity-related).
identity_attack = max(racism, sexism). The moderator decision is kept separately
to evaluate recommend() later.

In [ ]:
toxic_votes = df_rp[toxic_cols].sum(axis=1)
other_votes = df_rp[other_cols].sum(axis=1)
exclude = (df_rp["Reject Crowd"] == 1) & (toxic_votes <= other_votes)
print(f"Excluded comments: {exclude.sum()}")

df_rp_unified = pd.DataFrame({
    "text": df_rp["Text"],
    "toxicity": df_rp["Rejection Count Crowd"] / 5,
    "insult": df_rp["Insult Count Crowd"] / 5,
    "threat": df_rp["Threat Count Crowd"] / 5,
    "obscene": df_rp["Profanity Count Crowd"] / 5,
    "identity_attack": df_rp[["Racism Count Crowd", "Sexism Count Crowd"]].max(axis=1) / 5,
    "moderator_reject": df_rp["Reject Newspaper"],
    "lang": "de",
    "source": "rp_crowd",
})[~exclude]

print(df_rp_unified.shape)
(df_rp_unified[["toxicity", "insult", "threat", "obscene", "identity_attack"]] >= 0.5).mean().mul(100).round(2)

In [ ]:
df_ge = pd.concat([df_ge_train.assign(split="train"), df_ge_test.assign(split="test")])

df_ge_unified = pd.DataFrame({
    "text": df_ge["comment_text"],
    "toxicity": df_ge["Sub1_Toxic"].astype(float),
    "lang": "de",
    "source": "germeval2021",
    "official_split": df_ge["split"],
})

df_german = pd.concat([df_rp_unified, df_ge_unified], ignore_index=True)
print(df_german.shape)
print(f"Duplicate texts: {df_german['text'].duplicated().sum()}")
print(f"Texts appearing in both sources: {(df_german.groupby('text')['source'].nunique() > 1).sum()}")
df_german.isna().sum()

In [ ]:
df_rp_part = df_german[df_german["source"] == "rp_crowd"]
missing_cat = df_rp_part["insult"].isna()

print(f"RP comments without categories: {missing_cat.sum()}")
print("Toxicity of these comments:")
print(df_rp_part.loc[missing_cat, "toxicity"].value_counts())

print("\nRows without text:")
print(df_german.loc[df_german["text"].isna(), ["source", "toxicity"]])

## Cleaning the German data
- RP: missing category counts mean nobody rejected the comment (toxicity = 0 in all
  56,167 cases), so they are set to 0. GermEval categories stay NaN (never annotated).
- 2 RP comments without text are removed.
- Duplicate texts are removed before splitting (to avoid data leakage).
- GermEval anonymization placeholders (@USER, @MEDIUM, @MODERATOR) are removed,
  because they only occur in GermEval and act as a confounder for the source

In [ ]:
category_cols = ["insult", "threat", "obscene", "identity_attack"]

# RP only: missing categories mean "no rejection votes" -> 0
is_rp = df_german["source"] == "rp_crowd"
df_german.loc[is_rp, category_cols] = df_german.loc[is_rp, category_cols].fillna(0)

#Added later on after more analysis of PLACEHOLDERS such as @user

# GermEval: remove anonymization placeholders. They only occur in GermEval and would
# let a model recognize the source instead of toxicity (confounder).
df_german["text"] = df_german["text"].str.replace(r"@(USER|MEDIUM|MODERATOR)", "", regex=True).str.strip()

# Comments that consisted only of placeholders are now empty -> treat as missing
df_german["text"] = df_german["text"].replace("", pd.NA)

# Remove rows without text, then duplicate texts
df_german = df_german.dropna(subset=["text"])
df_german = df_german.drop_duplicates(subset="text")

print(df_german.shape)
df_german.isna().sum()

- Leading/trailing whitespace is stripped from all texts. This also revealed 47 RP
  duplicates that differed only in whitespace. Result: 88,129 comments
  (83,972 RP, 4,157 GermEval).

## Summary: preparing the German data

**1. Two sources**
- RP-Crowd (Rheinische Post): 85,000 news comments, each rated by 5 crowd workers,
  plus the decision of the newspaper's moderators.
- GermEval 2021: 4,188 Facebook comments, only a binary toxic label (35 % toxic).

**2. Crowd labels instead of moderator decisions**
Moderators and crowd rarely agree on rejected comments: only 21 % of the comments
rejected by moderators were also rejected by the crowd. Moderation follows house rules
(off-topic, spam, legal risks), which is not the same as toxicity. Therefore the crowd
votes are used as labels; the moderator decision is kept to evaluate recommend() later.

**3. Mapping to the unified format**
- toxicity = rejection votes / 5 (same idea as Civil Comments)
- insult, threat, obscene (= profanity) = votes / 5
- identity_attack = max(racism, sexism) / 5
- 538 comments rejected mainly for meta discussion or advertisement were excluded,
  because their rejection is not toxicity-related.

**4. Missing values mean different things**
- RP: categories were only asked when someone rejected a comment. Missing values
  (56,167 comments, all with toxicity = 0) therefore mean 0.
- GermEval: categories were never annotated. Missing values stay NaN ("unknown")
  and must not be treated as 0.

**5. Cleaning**
2 comments without text and 464 duplicate texts were removed, before any train/test
split. Result: 88,185 German comments (84,019 RP, 4,166 GermEval).

**Key finding:** Only about 34 RP comments are majority-rated as threats. German
threats cannot be learned from this data alone; the multilingual model has to
transfer this knowledge from the English data (to be tested in Phase 7).

## Train / validation / test split
RP: 80 / 10 / 10, stratified by toxicity >= 0.5.
GermEval: official test split kept; 10 % of the official training split used for validation.

In [ ]:
# RP: split into 80 % train and 20 % rest, then the rest into 10 % validation and 10 % test
df_rp_part = df_german[df_german["source"] == "rp_crowd"]

rp_train, rp_rest = train_test_split(
    df_rp_part, test_size=0.2, stratify=df_rp_part["toxicity"] >= 0.5, random_state=42
)
rp_val, rp_test = train_test_split(
    rp_rest, test_size=0.5, stratify=rp_rest["toxicity"] >= 0.5, random_state=42
)

# GermEval: keep the official test split, take 10 % of the training split for validation
df_ge_part = df_german[df_german["source"] == "germeval2021"]
ge_test = df_ge_part[df_ge_part["official_split"] == "test"]
ge_train_full = df_ge_part[df_ge_part["official_split"] == "train"]

ge_train, ge_val = train_test_split(
    ge_train_full, test_size=0.1, stratify=ge_train_full["toxicity"] >= 0.5, random_state=42
)

# Combine everything with a split column
df_german = pd.concat([
    rp_train.assign(split="train"),
    rp_val.assign(split="validation"),
    rp_test.assign(split="test"),
    ge_train.assign(split="train"),
    ge_val.assign(split="validation"),
    ge_test.assign(split="test"),
]).drop(columns="official_split")

# Check: size and toxic share per source and split
pd.crosstab(
    [df_german["source"], df_german["split"]],
    df_german["toxicity"] >= 0.5,
    margins=True,
)

In [ ]:
df_german.to_parquet("../data/processed/german_unified.parquet", index=False)
print(df_german.shape)
df_german["split"].value_counts()

**Result:** 88,185 German comments saved to `data/processed/german_unified.parquet`
with a `split` column. Toxic share per split: RP 4.7 % (identical in all splits),
GermEval ~35 % (official test 37 %). German threats in the test split are too few
for a reliable evaluation.

# Content analysis (training split only)
## Comment length

In [ ]:
df_de_train = df_german[df_german["split"] == "train"].copy()
df_de_train["length"] = df_de_train["text"].str.len()
df_de_train["is_toxic"] = df_de_train["toxicity"] >= 0.5

df_de_train.groupby("source")["length"].describe()

In [ ]:
print(df_de_train.groupby(["source", "is_toxic"])["length"].median())

for source, group in df_de_train.groupby("source"):
    group["length"].plot(kind="hist", bins=50, alpha=0.6, label=source)

plt.legend()
plt.xlabel("Characters")
plt.title("Comment length by source")
plt.show()

In [ ]:
print(df_de_train.groupby(["source", "is_toxic"])["length"].median())

for source, group in df_de_train.groupby("source"):
    group["length"].plot(
        kind="hist", bins=50, range=(0, 2000), density=True, alpha=0.6, label=source
    )

plt.legend()
plt.xlabel("Characters")
plt.ylabel("Density")
plt.title("Comment length by source")
plt.show()

**Findings (length):** RP has a hard limit of 500 characters (median 211).
GermEval (Facebook) comments are shorter (median 112) but have no limit (max 7,029).
Toxic comments are slightly shorter in RP (200 vs. 212) but slightly longer in GermEval
(120 vs. 106), so length is not a reliable signal, consistent with the English data.

## Most distinctive words in toxic comments

In [ ]:
vectorizer_de = CountVectorizer(min_df=20)
counts_de = vectorizer_de.fit_transform(df_de_train["text"])
words_de = vectorizer_de.get_feature_names_out()

is_toxic_de = df_de_train["is_toxic"].values
df_words_de = pd.DataFrame({
    "word": words_de,
    "toxic": counts_de[is_toxic_de].sum(axis=0).A1,
    "clean": counts_de[~is_toxic_de].sum(axis=0).A1,
})
df_words_de["toxic_share"] = df_words_de["toxic"] / (df_words_de["toxic"] + df_words_de["clean"])

print(f"Base rate (toxic share in training data): {is_toxic_de.mean():.1%}")
print(f"Vocabulary size: {len(words_de)}")
df_words_de[df_words_de["toxic"] >= 50].sort_values("toxic_share", ascending=False).head(20)

In [ ]:
# Do the anonymization placeholders only appear in GermEval?
for token in ["@USER", "@MEDIUM", "@MODERATOR"]:
    counts = df_de_train.groupby("source")["text"].apply(lambda s: s.str.contains(token).sum())
    print(token, dict(counts))

In [ ]:
# Lower threshold to reveal less frequent insults
df_words_de[df_words_de["toxic"] >= 20].sort_values("toxic_share", ascending=False).head(20)

In [ ]:
has_dumme = df_de_train["text"].str.contains(r"\bdumme\b", case=False)
has_dummheit = df_de_train["text"].str.contains(r"\bdummheit\b", case=False)

print(f"Comments with 'dumme':    {has_dumme.sum()}")
print(f"Comments with 'dummheit': {has_dummheit.sum()}")
print(f"Comments with both:       {(has_dumme & has_dummheit).sum()}")

df_de_train.loc[has_dumme & has_dummheit, "text"].head(5)

In [ ]:
df_de_train.loc[df_de_train["text"].str.contains(r"\bsch\b", case=False), "text"].head(5)

In [ ]:
# Show 25 characters before and after each standalone "sch"
context = df_de_train["text"].str.extract(r"(?i)(.{0,25}\bsch\b.{0,25})")[0].dropna()
print(f"Comments with a standalone 'sch': {len(context)}")
context.head(10).tolist()

**Findings (words):**
- GermEval anonymization placeholders (@USER, @MEDIUM) appeared as "toxic" words only
  because GermEval has a much higher toxic rate (35 %) than RP (5 %): a source confounder.
  Removed. Platform-specific words ("posts", "facebook") remain → evaluate per source.
- Insults and vulgar words appear with a lower threshold ("scheiße", "idioten", "pack").
- Many top words relate to migration and identity ("türken", "islam", "muslime",
  "migrationshintergrund") → bias risk, stronger than in the English data.
  "asylanten" is itself a derogatory term, unlike the neutral "flüchtlinge".
- Second-person address ("du", "euch", "ihr") is more frequent in toxic comments:
  attacks target people directly.
- 114 comments contain self-censored profanity ("Sch***ß", "sch..egal", "A.sch"),
  likely an adaptation to moderation. Word-level tokenization reduces them to "sch"
  → character n-grams in Phase 3.